Carga de librerías y definir ruta

In [9]:
import pandas as pd
import os
import re

# Definición de ruta
if os.path.exists('data'):
    PATH_DATA = 'data'
elif os.path.exists(os.path.join('..', 'data')):
    PATH_DATA = os.path.join('..', 'data')
else:
    PATH_DATA = 'data'

print(f"📂 Directorio de datos configurado en: {os.path.abspath(PATH_DATA)}")

📂 Directorio de datos configurado en: /Users/curro/Trabajo IA/Trabajo-Inteligencia-Artificial/data


ETL del archivo listings.csv

In [10]:
# Definición de columnas
final_cols = [
    'id', 'host_id', 'host_is_superhost', 'neighbourhood_cleansed', 
    'neighbourhood_group_cleansed', 'latitude', 'longitude', 
    'property_type', 'room_type', 'accommodates', 'bathrooms_text', 
    'bedrooms', 'price', 'minimum_nights', 'number_of_reviews', 
    'review_scores_rating', 'license', 'instant_bookable',
    'availability_365', 'calculated_host_listings_count', 
    'reviews_per_month', 'amenities'
]

ruta_listings = os.path.join(PATH_DATA, 'listings.csv')

# Carga y limpieza
if os.path.exists(ruta_listings):
    df_informacion = pd.read_csv(ruta_listings, usecols=final_cols, low_memory=False)
    
    # Limpieza de precio
    df_informacion['price'] = df_informacion['price'].replace(r'[\$,]', '', regex=True).astype(float)
    
    print(f"Dimensiones: {df_informacion.shape}")
else:
    print(f"Error: No se encuentra listings.csv en {ruta_listings}")

Dimensiones: (8215, 22)


ETL del archivo calendar.csv

In [11]:
cols_calendar = ['listing_id', 'date', 'available', 'minimum_nights', 'maximum_nights']
ruta_calendar = os.path.join(PATH_DATA, 'calendar.csv.zip')

if os.path.exists(ruta_calendar):
    df_calendar = pd.read_csv(ruta_calendar, usecols=cols_calendar, low_memory=False, compression='zip')
    
    df_calendar['date'] = pd.to_datetime(df_calendar['date'])
    df_calendar['available'] = df_calendar['available'].map({'t': True, 'f': False})
    df_calendar = df_calendar.sort_values(by=['listing_id', 'date'])
    
    print(f"Dimensiones: {df_calendar.shape}")
else:
    print(f"Error: No se encuentra calendar.csv.zip en {ruta_calendar}")

Dimensiones: (2998475, 5)


ETL del archivo de rentas

In [12]:
ruta_renta = os.path.join(PATH_DATA, 'renta_provincia_sevilla.csv')

if os.path.exists(ruta_renta):
    # Carga del CSV bruto del INE
    df_renta_raw = pd.read_csv(ruta_renta, sep=';', encoding='latin-1', decimal=',')
    
    # Transformación 1: Filtrado de Sevilla Capital (Código 41091)
    df_renta = df_renta_raw[
        (df_renta_raw['Secciones'].notna()) & 
        (df_renta_raw['Secciones'].astype(str).str.contains('^41091'))
    ].copy()
    
    # Transformación 2: Limpieza del formato numérico
    if df_renta['Total'].dtype == 'object':
        df_renta['Total'] = df_renta['Total'].str.replace('.', '', regex=False).astype(float)
        
    print(f"Secciones resultantes: {len(df_renta)}")
else:
    print(f"Error: No se encuentra la renta en {ruta_renta}")

Secciones resultantes: 541


Guardado de los dataframes limpios para su posterior uso

In [13]:
# 1. Guardamos listings limpio
df_informacion.to_csv(os.path.join(PATH_DATA, 'listings_limpio.csv'), index=False)
print("listings_limpio.csv guardado.")

# 2. Guardamos calendar limpio (comprimido)
df_calendar.to_csv(os.path.join(PATH_DATA, 'calendar_limpio.csv.zip'), index=False, compression='zip')
print("calendar_limpio.csv.zip guardado.")

# 3. Guardamos la renta limpia 
df_renta.to_csv(os.path.join(PATH_DATA, 'renta_sevilla_capital_limpio.csv'), index=False)
print("renta_sevilla_capital_limpio.csv guardado.")



listings_limpio.csv guardado.
calendar_limpio.csv.zip guardado.
renta_sevilla_capital_limpio.csv guardado.
